In [4]:
from typing import Dict, List, Annotated
import numpy as np
from sklearn.cluster import MiniBatchKMeans
import pickle
import numpy as np
n_clusters_1 = 2
n_clusters_2 = 3

vectors = np.random.randint(0, 1000, (100, 7))

def _write_vectors_to_file(vectors: np.ndarray) -> None:
    mmap_vectors = np.memmap("dbgdeda.dat", dtype=np.int32, mode='w+', shape=vectors.shape)
    mmap_vectors[:] = vectors[:]
    mmap_vectors.flush()


def get_n_random_rows(indices) -> np.ndarray:
    try:
        min_idx = indices.min()
        max_idx = indices.max()
        offset = min_idx * 7 * np.dtype(np.int32).itemsize
        n_rows = max_idx - min_idx + 1
        mmap_vector = np.memmap(
            "dbgdeda.dat",
            dtype=np.int32,
            mode='r',
            shape=(n_rows, 7),
            offset=offset
        )
        relative_indices = indices - min_idx
        # print(mmap_vector[relative_indices])
        return np.array(mmap_vector[relative_indices])
    except Exception as e:
        return f"An error occurred: {e}"
    

def _build_index():
    kmeans = MiniBatchKMeans(n_clusters=n_clusters_1, batch_size=10, max_iter=200)
    kmeans.fit(vectors)

    labels = kmeans.predict(vectors)
    centroids = kmeans.cluster_centers_

    cluster_mapping = {tuple(centroid): [] for centroid in centroids}

    for vector_id, label in enumerate(labels):
        centroid_key = tuple(centroids[label])
        cluster_mapping[centroid_key].append(vector_id)

    # print("Labels:", labels)
    # print("Centroids:", centroids)
    # print("Cluster Mapping:", cluster_mapping)


    with open("centroidstry.dat", 'wb') as index_file:
        pickle.dump(centroids, index_file)

    # Second level
    kmeans_2nd_level = MiniBatchKMeans(n_clusters=n_clusters_2, batch_size=10, max_iter=200)

    cluster_mapping_1 = cluster_mapping

    for i,centroid_1 in enumerate(cluster_mapping_1.keys()):

        cluster_vector_ids = cluster_mapping_1[centroid_1]
        cluster_vector_ids_np = np.array(cluster_vector_ids)

        cluster_vectors = get_n_random_rows(cluster_vector_ids_np)

        labels = kmeans_2nd_level.fit_predict(cluster_vectors)
        centroids_2 = kmeans_2nd_level.cluster_centers_
        # print("Labels:", labels)
        print("Centroids2:", centroids_2)

        cluster_mapping_2 = {tuple(centroid_2): [] for centroid_2 in centroids_2}

        for vector_id, label in zip(cluster_vector_ids,labels):
            centroid_2_key = tuple(centroids_2[label])
            cluster_mapping_2[centroid_2_key].append(vector_id)    
    

        print("FINALLLLLLLLLL",cluster_mapping_2)

        with open(f"index2try_{i}.dat", 'wb') as index_file:
            pickle.dump(cluster_mapping_2, index_file)

_write_vectors_to_file(vectors)

_build_index()

Centroids2: [[483.76666667 521.78333333 144.11666667 264.33333333 413.41666667
  826.66666667 413.56666667]
 [763.32758621 616.15517241 532.85344828 467.31896552 527.82758621
  590.93103448 540.9137931 ]
 [602.76190476 353.60714286 456.75       197.52380952 320.47619048
  239.95238095 556.89285714]]
FINALLLLLLLLLL {(483.76666666666665, 521.7833333333332, 144.11666666666662, 264.33333333333337, 413.4166666666667, 826.6666666666666, 413.5666666666666): [18, 28, 35, 38, 43, 47, 53, 54, 60, 62, 64, 65, 69, 97], (763.3275862068965, 616.155172413793, 532.853448275862, 467.3189655172415, 527.8275862068966, 590.9310344827586, 540.9137931034486): [1, 2, 11, 14, 15, 16, 19, 20, 22, 24, 25, 30, 56, 61, 67, 68, 71, 75, 78, 79, 83, 85, 87, 88, 90, 96], (602.7619047619047, 353.6071428571428, 456.75, 197.52380952380943, 320.47619047619037, 239.952380952381, 556.8928571428571): [3, 4, 8, 23, 31, 32, 33, 34, 40, 44, 48, 49, 66, 70, 72, 74, 89, 91, 93]}
Centroids2: [[258.97260274 427.5890411  595.712328